# Sunrise Social Club: Predictive Modeling

**Objective:** Predict beverage demand to support inventory planning and event preparation.

### Loading the Data and Overview

Out of the gate, our biggest business problem was figuring out how much cold brew, matcha, and lemonade to batch. Sometimes we would batch too little and sell out too quickly, and sometimes we would batch too much and have product go to waste. This predictive model aims to predict how much of each drink type we'll sell at a given market/event, and equate that to how many gallons to batch.

In [1079]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [1080]:
df = pd.read_csv("../data/processed/sales_merged.csv")
drink_df = df[df['Item'].isin(['Matcha Latte','Cold Brew','Lemonade'])]

In [1081]:
model_df = (
    drink_df.groupby(
        ["Date", "Event_Type", "Location", "Item", "Avg_Temp","Weather_Condition"]
    )["Qty"]
    .sum()
    .reset_index()
)
model_df.head()

,Date,Event_Type,Location,Item,Avg_Temp,Weather_Condition,Qty
0,2026-05-23,Popup,Kill Devil Hills,Matcha Latte,77.0,Partly Cloudy,68.0
1,2026-05-30,Market,Manteo,Cold Brew,79.0,Sunny,51.0
2,2026-05-30,Market,Manteo,Lemonade,79.0,Sunny,21.0
3,2026-05-30,Market,Manteo,Matcha Latte,79.0,Sunny,87.0
4,2026-06-06,Market,Manteo,Cold Brew,86.0,Sunny,46.0


In [1082]:
model_df['Date']=pd.to_datetime(model_df['Date'])
model_df['Month']=model_df['Date'].dt.month
model_df['Weekend']= model_df['Date'].dt.dayofweek >=5

### Feature Engineering

In [1083]:
X = model_df[[
    "Event_Type",
    "Location",
    "Avg_Temp",
    "Weather_Condition",
    "Month",
    "Weekend",
    "Item"
]]
y=model_df['Qty']

To improve predictive performance, several features were derived from the raw data:

- Month captures seasonal demand.
- Weekend indicates increased customer traffic.
- Event type and location capture differences between markets.
- Weather variables account for temperature-driven purchasing behavior.

### Train/Test Split

In [1084]:
X=pd.get_dummies(X,drop_first=True)

In [1085]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Baseline Model

In [1086]:
baseline_pred = np.full_like(y_test, y_train.mean())

In [1087]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
mae_baseline = mean_absolute_error(y_test, baseline_pred)
mse_baseline = mean_squared_error(y_test, baseline_pred)
rmse_baseline = np.sqrt(mse_baseline)

### Linear Regression

In [1088]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

In [1089]:
mae_lr = mean_absolute_error(y_test, pred_lr)
rmse_lr = root_mean_squared_error(y_test, pred_lr)

### Random Forest

In [1090]:
rf=RandomForestRegressor(random_state=42)
rf.fit(X_train,y_train)
pred_rf=rf.predict(X_test)

In [1091]:
mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = root_mean_squared_error(y_test, pred_rf)

### Model Evaluation

In [1092]:
results = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression", "Random Forest"],
    "MAE": [mae_baseline, mae_lr, mae_rf],
    "RMSE": [rmse_baseline, rmse_lr, rmse_rf]
})
results

,Model,MAE,RMSE
0,Baseline,17.737903,21.827316
1,Linear Regression,15.827145,18.320685
2,Random Forest,16.068750,18.563004


Although linear regression achieved slightly lower MAE and RMSE, the difference was less than a quarter of a drink per prediction. A random forest was selected for deployment because it can capture nonlinear relationships between weather, event characteristics, and beverage demand. As additional sales data are collected, these nonlinear patterns are expected to become more informative.

### Inventory Calculations

In [1095]:
grams_12oz = {
    "Lemonade": 200,
    "Cold Brew": 100,
    "Matcha Latte": 40
}

grams_16oz = {
    "Lemonade": 300,
    "Cold Brew": 180,
    "Matcha Latte": 50
}

In [1096]:
def compute_effective_grams():

    mix_16 = 0.7
    mix_12 = 0.3

    effective_grams = {}

    for item in grams_12oz.keys():
        effective_grams[item] = (
            mix_16 * grams_16oz[item] +
            mix_12 * grams_12oz[item]
        )

    return effective_grams

In [1097]:
def inventory_calculator(pred_df):

    effective_grams = compute_effective_grams()

    GRAMS_PER_GALLON = 3300

    pred_df["Grams"] = pred_df.apply(
        lambda row: row["Predicted_Qty"] * effective_grams[row["Item"]],
        axis=1
    )

    pred_df["Gallons"] = pred_df["Grams"] / GRAMS_PER_GALLON

    return pred_df

In [1098]:
event_input_raw = pd.DataFrame({
    "Event_Type": ["Market"],
    "Location": ["Manteo"],
    "Avg_Temp": [85],
    "Weather_Condition": ["Sunny"],
    "Month": [6],
    "Weekend": [True],
})

In [1099]:
def build_event_grid(event_input_raw):

    items = ["Matcha Latte", "Cold Brew", "Lemonade"]

    rows = []

    for item in items:
        row = event_input_raw.copy()
        row["Item"] = item
        rows.append(row)

    return pd.concat(rows, ignore_index=True)

In [1100]:
event_grid = build_event_grid(event_input_raw)

event_encoded = pd.get_dummies(event_grid)
event_encoded = event_encoded.reindex(columns=X.columns, fill_value=0)

event_grid["Predicted_Qty"] = rf.predict(event_encoded)

In [1101]:
result = inventory_calculator(event_grid)

In [1102]:
result

,Event_Type,Location,Avg_Temp,Weather_Condition,Month,Weekend,Item,Predicted_Qty,Grams,Gallons
0,Market,Manteo,85,Sunny,6,True,Matcha Latte,64.08,3011.76,0.912655
1,Market,Manteo,85,Sunny,6,True,Cold Brew,52.93,8257.08,2.502145
2,Market,Manteo,85,Sunny,6,True,Lemonade,38.29,10338.30,3.132818
